# SYEnet Palmprint Recognition

Portfolio version focused on the core AI pipeline: dataset indexing, palm ROI preprocessing, 9-patch feature extraction, SYEnet CNN training and few-shot recognition with Collaborative Representation Classification (CRC).

> Dataset files are not included. Update the paths in `Config` before running.

## 1. Dependencies and imports

In [ ]:
%pip install -q pandas scikit-learn pillow tqdm opencv-python-headless


In [ ]:
import os
import re
import json
import math
import time
import random
import warnings
import hashlib
import gc
from pathlib import Path
from dataclasses import dataclass, asdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. Configuration

In [ ]:
@dataclass
class Config:
    # Update these paths after downloading the datasets.
    project_root: str = "./artifacts"
    casia_root: str = "./data/CASIA"
    tongji_root: str = "./data/Tongji/ROI"

    baseline_profile: str = "paper_stable"

    build_patch_cache: bool = True
    patch_cache_root: str = "./artifacts/patch_cache"
    patch_cache_version: str = "v5_paper_baseline_roi128_9patch"

    # SYEnet input and 3x3 overlapping patch setup.
    image_size: int = 128
    patch_canvas_size: int = 128
    patch_crop_size: int = 64
    patch_stride: int = 32
    use_9_patches: bool = True

    # FC2 produces an 84-dimensional embedding.
    feature_aggregation: str = "mean"

    # CASIA hand alignment and ROI extraction.
    casia_roi_scale: float = 2.70
    casia_roi_y_shift: float = 0.08
    casia_align_rotation: bool = True

    apply_clahe: bool = False
    use_intensity_augmentation: bool = False

    epochs: int = 30
    batch_size: int = 32
    dropout: float = 0.50

    num_workers: int = min(4, os.cpu_count() or 2)
    prefetch_factor: int = 4
    persistent_workers: bool = True
    pin_memory: bool = True

    optimizer_name: str = "adamw"
    learning_rate: float = 1.0e-3
    momentum: float = 0.9
    weight_decay: float = 1.0e-4
    scheduler_name: str = "plateau"
    scheduler_factor: float = 0.5
    scheduler_patience: int = 3
    min_learning_rate: float = 1.0e-6

    label_smoothing: float = 0.0
    gradient_clip: float = 5.0

    crc_lambda: float = 1e-3
    repeats: int = 10

    use_amp: bool = True
    channels_last: bool = True
    seed: int = 42

    quick_run: bool = False
    quick_num_classes: int = 40
    quick_epochs: int = 3


def apply_baseline_profile(cfg):
    if cfg.baseline_profile == "paper_strict":
        cfg.optimizer_name = "sgd"
        cfg.learning_rate = 5.0e-2
        cfg.momentum = 0.9
        cfg.weight_decay = 0.0
        cfg.scheduler_name = "none"
        cfg.gradient_clip = 0.0

    elif cfg.baseline_profile == "paper_stable":
        cfg.optimizer_name = "adamw"
        cfg.learning_rate = 1.0e-3
        cfg.weight_decay = 1.0e-4
        cfg.scheduler_name = "plateau"
        cfg.gradient_clip = 5.0

    else:
        raise ValueError("baseline_profile must be 'paper_stable' or 'paper_strict'.")

    return cfg


CFG = apply_baseline_profile(Config())

PROJECT_ROOT = Path(CFG.project_root)
CASIA_ROOT = Path(CFG.casia_root)
TONGJI_ROOT = Path(CFG.tongji_root)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
Path(CFG.patch_cache_root).mkdir(parents=True, exist_ok=True)


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(CFG.seed)


## 3. Dataset indexing

In [ ]:
IMAGE_EXTENSIONS = {".bmp", ".jpg", ".jpeg", ".png", ".tif", ".tiff"}


def list_images(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset not found: {root}")

    files = sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not files:
        raise ValueError(f"No images found in: {root}")

    return files


In [ ]:
def detect_session(path):
    text = str(path).lower().replace("\\", "/")
    if any(x in text for x in ["session1", "session_1", "session 1", "/s1/"]):
        return "session1"
    if any(x in text for x in ["session2", "session_2", "session 2", "/s2/"]):
        return "session2"
    return "unknown"


def last_integer(text):
    values = re.findall(r"\d+", text)
    return int(values[-1]) if values else None


def build_tongji_index(root):
    files = list_images(root)
    has_more_than_one_session_block = len(files) > 6000
    rows = []

    for path in files:
        number = last_integer(path.stem)
        if number is None:
            continue

        session = detect_session(path)

        if 1 <= number <= 12000:
            if number > 6000:
                local_number = number - 6000
                if session == "unknown":
                    session = "session2"
            else:
                local_number = number
                if session == "unknown" and has_more_than_one_session_block:
                    session = "session1"

            palm_id = (local_number - 1) // 10
            sample = (local_number - 1) % 10
        else:
            palm_id = path.parent.name
            sample = number

        rows.append({
            "path": str(path),
            "label_raw": f"tongji_{palm_id}",
            "session": session,
            "sample": sample,
            "dataset": "Tongji",
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Unable to build Tongji index.")

    return df


In [ ]:
def filename_tokens(path):
    return [x for x in re.split(r"[_\-\s.]+", path.stem) if x]


HAND_MAP = {
    "l": "L", "left": "L", "lh": "L",
    "r": "R", "right": "R", "rh": "R"
}


def casia_auto_person_hand(path, root):
    tokens = filename_tokens(path)
    low = [t.lower() for t in tokens]

    hand = None
    for token in low:
        if token in HAND_MAP:
            hand = HAND_MAP[token]
            break

    numeric_tokens = [t for t in tokens if re.fullmatch(r"\d+", t)]
    person = numeric_tokens[0] if numeric_tokens else None

    if person is not None and hand is not None:
        return f"{int(person):04d}_{hand}"

    parent = path.parent.name
    parent_hand = None

    for token in re.split(r"[_\-\s.]+", parent.lower()):
        if token in HAND_MAP:
            parent_hand = HAND_MAP[token]
            break

    parent_nums = re.findall(r"\d+", parent)
    if parent_nums and parent_hand:
        return f"{int(parent_nums[0]):04d}_{parent_hand}"

    if len(tokens) > 1:
        return "_".join(tokens[:-1])

    return tokens[0]


def build_casia_index(root):
    rows = []

    for path in list_images(root):
        label = casia_auto_person_hand(path, root)
        rows.append({
            "path": str(path),
            "label_raw": f"casia_{label}",
            "session": detect_session(path),
            "sample": last_integer(path.stem),
            "dataset": "CASIA",
        })

    return pd.DataFrame(rows)


def filter_small_classes(df, minimum=5):
    counts = df.groupby("label_raw").size()
    valid = counts[counts >= minimum].index
    return df[df["label_raw"].isin(valid)].copy().reset_index(drop=True)


def quick_subset(df, num_classes, seed=42):
    labels = sorted(df["label_raw"].unique())

    if len(labels) <= num_classes:
        return df.copy().reset_index(drop=True)

    rng = np.random.default_rng(seed)
    selected = rng.choice(labels, size=num_classes, replace=False)

    return df[df["label_raw"].isin(selected)].copy().reset_index(drop=True)


In [ ]:
casia_df = filter_small_classes(build_casia_index(CASIA_ROOT), minimum=5)
tongji_df = filter_small_classes(build_tongji_index(TONGJI_ROOT), minimum=5)

if CFG.quick_run:
    casia_work_df = quick_subset(casia_df, CFG.quick_num_classes, CFG.seed)
    tongji_work_df = quick_subset(tongji_df, CFG.quick_num_classes, CFG.seed)
else:
    casia_work_df = casia_df.copy()
    tongji_work_df = tongji_df.copy()


## 4. Palm ROI preprocessing and 9-patch extraction

In [ ]:

def pil_gray_to_uint8(img):
    img = ImageOps.exif_transpose(img).convert("L")
    return np.asarray(img, dtype=np.uint8)


def enhance_roi_uint8(arr, apply_clahe=False):
    if not apply_clahe:
        return arr
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(arr)


def fallback_center_palm(gray):
    h, w = gray.shape
    side = int(min(h, w) * 0.58)
    cx = w // 2
    cy = int(h * 0.58)
    x1 = min(max(0, cx - side // 2), max(0, w - side))
    y1 = min(max(0, cy - side // 2), max(0, h - side))
    return gray[y1:y1 + side, x1:x1 + side]


def largest_hand_mask(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(
        blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Giữ vùng bàn tay là foreground trắng.
    if mask.mean() > 127:
        mask = 255 - mask

    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n_labels <= 1:
        return None

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return np.where(labels == largest, 255, 0).astype(np.uint8)


def rotate_bound(gray, mask, angle_deg):
    h, w = gray.shape
    center = (w / 2.0, h / 2.0)
    matrix = cv2.getRotationMatrix2D(center, angle_deg, 1.0)

    cos = abs(matrix[0, 0])
    sin = abs(matrix[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)

    matrix[0, 2] += new_w / 2.0 - center[0]
    matrix[1, 2] += new_h / 2.0 - center[1]

    gray_rot = cv2.warpAffine(
        gray, matrix, (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    mask_rot = cv2.warpAffine(
        mask, matrix, (new_w, new_h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    return gray_rot, mask_rot


def align_hand_by_pca(gray, hand_mask):
    ys, xs = np.where(hand_mask > 0)
    if len(xs) < 100:
        return gray, hand_mask

    points = np.column_stack([xs, ys]).astype(np.float32)
    _, eigenvectors = cv2.PCACompute(points, mean=None, maxComponents=2)
    vx, vy = eigenvectors[0]
    principal_angle = np.degrees(np.arctan2(vy, vx))

    # Đưa trục dài của bàn tay về phương thẳng đứng.
    rotation = 90.0 - principal_angle
    while rotation > 90:
        rotation -= 180
    while rotation < -90:
        rotation += 180

    return rotate_bound(gray, hand_mask, rotation)


def extract_casia_roi(img, cfg):
    """
    Xấp xỉ PREnet khi không có checkpoint/nhãn YOLO:
    tách bàn tay -> căn chỉnh hướng -> distance transform -> crop ROI.
    """
    gray = pil_gray_to_uint8(img)
    hand_mask = largest_hand_mask(gray)

    if hand_mask is None:
        roi = fallback_center_palm(gray)
    else:
        if cfg.casia_align_rotation:
            gray, hand_mask = align_hand_by_pca(gray, hand_mask)
            # Làm sạch lại mask sau phép xoay.
            hand_mask = largest_hand_mask(gray)
            if hand_mask is None:
                roi = fallback_center_palm(gray)
                roi = enhance_roi_uint8(roi, cfg.apply_clahe)
                return Image.fromarray(roi)

        dist = cv2.distanceTransform(hand_mask, cv2.DIST_L2, 5)
        _, radius, _, center = cv2.minMaxLoc(dist)
        cx, cy = center
        cy = int(cy + cfg.casia_roi_y_shift * radius)
        side = int(max(64, cfg.casia_roi_scale * radius))

        h, w = gray.shape
        side = min(side, h, w)
        x1 = min(max(0, int(round(cx - side / 2))), max(0, w - side))
        y1 = min(max(0, int(round(cy - side / 2))), max(0, h - side))
        roi = gray[y1:y1 + side, x1:x1 + side]

        if roi.size == 0 or min(roi.shape) < 32:
            roi = fallback_center_palm(gray)

    roi = enhance_roi_uint8(roi, cfg.apply_clahe)
    return Image.fromarray(roi)


def prepare_roi(img, dataset_name, cfg):
    name = str(dataset_name).strip().lower()
    if name == "casia":
        return extract_casia_roi(img, cfg)
    if name == "tongji":
        arr = pil_gray_to_uint8(img)
        arr = enhance_roi_uint8(arr, cfg.apply_clahe)
        return Image.fromarray(arr)
    raise ValueError(f"Dataset chưa được hỗ trợ: {dataset_name!r}")


def make_patches_uint8(img, dataset_name, cfg):
    roi = prepare_roi(img, dataset_name, cfg)
    roi = roi.resize(
        (cfg.patch_canvas_size, cfg.patch_canvas_size),
        Image.Resampling.BILINEAR
    )

    if not cfg.use_9_patches:
        arr = np.asarray(
            roi.resize((cfg.image_size, cfg.image_size), Image.Resampling.BILINEAR),
            dtype=np.uint8
        )
        return arr[None, ...]

    expected_canvas = cfg.patch_crop_size + 2 * cfg.patch_stride
    if cfg.patch_canvas_size != expected_canvas:
        raise ValueError(
            "patch_canvas_size phải bằng patch_crop_size + 2*patch_stride."
        )

    offsets = [0, cfg.patch_stride, 2 * cfg.patch_stride]
    patches = []
    for top in offsets:
        for left in offsets:
            patch = roi.crop((
                left,
                top,
                left + cfg.patch_crop_size,
                top + cfg.patch_crop_size
            ))
            # Backbone của bài báo nhận đầu vào 128×128.
            patch = patch.resize(
                (cfg.image_size, cfg.image_size),
                Image.Resampling.BILINEAR
            )
            patches.append(np.asarray(patch, dtype=np.uint8))

    return np.stack(patches, axis=0)


def make_patches(img, dataset_name, cfg):
    arr = make_patches_uint8(img, dataset_name, cfg)
    x = torch.from_numpy(arr.copy()).float().div_(255.0).unsqueeze(1)
    return x


In [ ]:

def add_noise(x, mode=None, seed=0):
    if mode is None:
        return x
    g = torch.Generator().manual_seed(seed)
    if mode == "sp":
        rnd = torch.rand(x.shape, generator=g)
        out = x.clone()
        out[rnd < 0.01] = 0.0
        out[(rnd >= 0.01) & (rnd < 0.02)] = 1.0
        return out
    if mode == "gaussian":
        noise = torch.randn(x.shape, generator=g) * 0.1
        return torch.clamp(x + noise, 0.0, 1.0)
    raise ValueError("noise mode không hợp lệ")


class PalmDataset(Dataset):
    def __init__(
        self, df, cfg, label_to_idx=None,
        noise_mode=None, training=False
    ):
        self.df = df.reset_index(drop=True).copy()
        self.cfg = cfg
        self.noise_mode = noise_mode
        self.training = training
        self._cache = None

        if label_to_idx is None:
            labels = sorted(self.df["label_raw"].unique())
            label_to_idx = {v: i for i, v in enumerate(labels)}
        self.label_to_idx = label_to_idx

        self.cache_path = None
        if "patch_cache_path" in self.df.columns:
            values = self.df["patch_cache_path"].dropna().unique()
            if len(values) == 1:
                self.cache_path = values[0]

    def __len__(self):
        return len(self.df)

    def _open_cache(self):
        if self._cache is None and self.cache_path is not None:
            self._cache = np.load(self.cache_path, mmap_mode="r")
        return self._cache

    def __getitem__(self, index):
        row = self.df.iloc[index]

        if self.cache_path is not None:
            cache = self._open_cache()
            arr = np.asarray(
                cache[int(row["patch_cache_index"])],
                dtype=np.float32
            )
            patches = torch.from_numpy(arr.copy()).div_(255.0).unsqueeze(1)
        else:
            with Image.open(row["path"]) as img:
                patches = make_patches(
                    img.copy(),
                    dataset_name=row["dataset"],
                    cfg=self.cfg
                )

        # Tắt mặc định vì bài báo không mô tả augmentation cường độ.
        if self.training and self.cfg.use_intensity_augmentation:
            gain = 0.90 + 0.20 * torch.rand(1).item()
            bias = -0.05 + 0.10 * torch.rand(1).item()
            patches = torch.clamp(patches * gain + bias, 0.0, 1.0)

        patches = add_noise(
            patches,
            self.noise_mode,
            self.cfg.seed + index
        )

        # Giữ chuẩn hóa cố định, áp dụng giống nhau ở train và test.
        patches = (patches - 0.5) / 0.5

        label = self.label_to_idx[row["label_raw"]]
        return (
            patches,
            torch.tensor(label, dtype=torch.long),
            row["path"]
        )


## 5. Optional patch cache

In [ ]:
def cache_signature(df, name, cfg):
    payload = {
        "name": name,
        "version": cfg.patch_cache_version,
        "n": len(df),
        "paths": df["path"].tolist(),
        "image_size": cfg.image_size,
        "canvas": cfg.patch_canvas_size,
        "crop": cfg.patch_crop_size,
        "stride": cfg.patch_stride,
        "clahe": cfg.apply_clahe,
        "casia_scale": cfg.casia_roi_scale,
        "casia_y_shift": cfg.casia_roi_y_shift,
        "casia_align_rotation": cfg.casia_align_rotation,
        "feature_aggregation": cfg.feature_aggregation,
    }
    raw = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha1(raw).hexdigest()[:12]


def attach_patch_cache(df, name, cfg, force=False):
    df = df.reset_index(drop=True).copy()

    if not cfg.build_patch_cache:
        return df

    signature = cache_signature(df, name, cfg)
    cache_dir = Path(cfg.patch_cache_root)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{name.lower()}_{signature}.npy"
    meta_path = cache_dir / f"{name.lower()}_{signature}.json"

    shape = (
        len(df),
        9 if cfg.use_9_patches else 1,
        cfg.image_size,
        cfg.image_size
    )

    valid_existing = cache_path.exists() and meta_path.exists()
    if valid_existing and not force:
        try:
            existing = np.load(cache_path, mmap_mode="r")
            valid_existing = tuple(existing.shape) == shape
        except Exception:
            valid_existing = False

    if not valid_existing or force:
        print(f"Đang tạo patch cache {name}: {cache_path}")
        mmap = np.lib.format.open_memmap(
            cache_path, mode="w+", dtype=np.uint8, shape=shape
        )

        for i, row in tqdm(df.iterrows(), total=len(df)):
            with Image.open(row["path"]) as img:
                mmap[i] = make_patches_uint8(
                    img.copy(), row["dataset"], cfg
                )
        mmap.flush()
        del mmap

        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(
                {"shape": shape, "signature": signature, "version": cfg.patch_cache_version},
                f, ensure_ascii=False, indent=2
            )
    else:
        print(f"Dùng lại patch cache {name}: {cache_path}")

    df["patch_cache_path"] = str(cache_path)
    df["patch_cache_index"] = np.arange(len(df), dtype=np.int64)
    return df

casia_work_df = attach_patch_cache(casia_work_df, "CASIA", CFG)
tongji_work_df = attach_patch_cache(tongji_work_df, "Tongji", CFG)


## 6. SYEnet CNN

In [ ]:

class ConvReLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=stride, padding=0, bias=True
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.conv(x))


class SYEnetBackbone(nn.Module):
    """
    Cấu trúc theo Hình 6:
    4 convolution, 3 max-pooling, FC1=120, FC2=84.
    Không thêm BatchNorm/LayerNorm vì bài báo không mô tả các lớp này.
    ReLU dùng ở mọi lớp trừ lớp phân loại cuối.
    """
    def __init__(self, dropout=0.50):
        super().__init__()
        self.conv1 = ConvReLU(1, 12, 5, stride=2)
        self.conv2 = ConvReLU(12, 40, 3, stride=1)
        self.conv3 = ConvReLU(40, 32, 5, stride=1)
        self.conv4 = ConvReLU(32, 40, 5, stride=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(40, 120)
        self.fc2 = nn.Linear(120, 84)
        self.dropout = nn.Dropout(dropout)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(
                module.weight, mode="fan_out", nonlinearity="relu"
            )
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Linear):
            nn.init.kaiming_uniform_(
                module.weight, a=math.sqrt(5)
            )
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.pool(self.conv1(x))
        x = self.pool(self.conv2(x))
        x = self.pool(self.conv3(x))
        x = self.conv4(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x), inplace=True)
        x = self.dropout(x)
        x = F.relu(self.fc2(x), inplace=True)
        return x


class SYEnet(nn.Module):
    def __init__(self, num_classes, cfg):
        super().__init__()
        self.cfg = cfg
        self.num_patches = 9 if cfg.use_9_patches else 1
        self.patch_feature_dim = 84
        self.feature_dim = 84

        self.backbone = SYEnetBackbone(cfg.dropout)
        # FC3/softmax head; CrossEntropyLoss nhận logits trực tiếp.
        self.classifier = nn.Linear(self.feature_dim, num_classes)

    def aggregate_embeddings(self, emb):
        # Bài báo không công bố phép gộp. Mean giữ đúng kích thước FC2=84
        # và tránh tạo classifier 756-D không xuất hiện trong Hình 6.
        return emb.mean(dim=1)

    def forward(
        self, patches, return_features=False,
        return_patch_embeddings=False
    ):
        if patches.ndim != 5:
            raise ValueError(
                f"Đầu vào phải [B,P,C,H,W], nhận {tuple(patches.shape)}"
            )

        b, p, c, h, w = patches.shape
        if p != self.num_patches:
            raise ValueError(
                f"Yêu cầu {self.num_patches} patch nhưng nhận {p}."
            )

        x = patches.reshape(b * p, c, h, w)
        if self.cfg.channels_last and x.is_cuda:
            x = x.contiguous(memory_format=torch.channels_last)

        emb = self.backbone(x)
        emb = emb.reshape(b, p, self.patch_feature_dim)
        feat = self.aggregate_embeddings(emb)
        logits = self.classifier(feat)

        if return_patch_embeddings:
            return logits, emb
        if return_features:
            return logits, feat
        return logits

    @torch.no_grad()
    def extract_features(self, patches):
        _, feat = self.forward(patches, return_features=True)
        return F.normalize(feat.float(), p=2, dim=1)


## 7. Training pipeline

In [ ]:

def split_source(df, val_ratio=0.2, seed=42):
    train_parts, val_parts = [], []
    for label, group in df.groupby("label_raw"):
        group = group.sample(frac=1, random_state=seed)
        n_val = max(1, int(round(len(group) * val_ratio)))
        if len(group) - n_val < 1:
            n_val = len(group) - 1
        val_parts.append(group.iloc[:n_val])
        train_parts.append(group.iloc[n_val:])

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)

    if set(train_df["label_raw"]) != set(val_df["label_raw"]):
        raise RuntimeError("Train/validation không chứa cùng tập lớp.")

    return train_df, val_df


def make_loader(dataset, cfg, shuffle=False, drop_last=False, batch_size=None):
    bs = batch_size or cfg.batch_size
    kwargs = dict(
        dataset=dataset,
        batch_size=bs,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory and torch.cuda.is_available(),
        drop_last=drop_last
    )
    if cfg.num_workers > 0:
        kwargs.update(
            persistent_workers=cfg.persistent_workers,
            prefetch_factor=cfg.prefetch_factor
        )
    return DataLoader(**kwargs)


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)


def run_epoch(
    model, loader, criterion,
    optimizer=None, scaler=None,
    gradient_clip=None
):
    training = optimizer is not None
    model.train(training)

    total_loss = total_correct = total_top5 = total = 0
    predicted_classes = set()

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for patches, labels, _ in tqdm(loader, leave=False):
            patches = patches.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            amp_enabled = CFG.use_amp and DEVICE.type == "cuda"
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=amp_enabled
            ):
                logits = model(patches)
                loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn: {loss.item()}")

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)

                if gradient_clip is not None and gradient_clip > 0:
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), gradient_clip
                    )

                scaler.step(optimizer)
                scaler.update()

            predictions = logits.argmax(1)
            predicted_classes.update(predictions.detach().cpu().tolist())

            k = min(5, logits.shape[1])
            topk = logits.topk(k, dim=1).indices
            top5_correct = topk.eq(labels[:, None]).any(dim=1).sum().item()

            total_loss += loss.detach().item() * labels.size(0)
            total_correct += (predictions == labels).sum().item()
            total_top5 += top5_correct
            total += labels.size(0)

    return {
        "loss": total_loss / total,
        "top1": total_correct / total,
        "top5": total_top5 / total,
        "predicted_classes": len(predicted_classes)
    }


In [ ]:

def build_optimizer(model, cfg):
    if cfg.optimizer_name.lower() == "adamw":
        return torch.optim.AdamW(
            model.parameters(),
            lr=cfg.learning_rate,
            weight_decay=cfg.weight_decay,
            betas=(0.9, 0.999)
        )

    if cfg.optimizer_name.lower() == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=cfg.learning_rate,
            momentum=cfg.momentum,
            weight_decay=cfg.weight_decay,
            nesterov=False
        )

    raise ValueError(cfg.optimizer_name)


def build_scheduler(optimizer, cfg):
    if cfg.scheduler_name == "none":
        return None
    if cfg.scheduler_name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=cfg.scheduler_factor,
            patience=cfg.scheduler_patience,
            min_lr=cfg.min_learning_rate
        )
    raise ValueError(cfg.scheduler_name)


def train_source(source_df, source_name, cfg):
    train_df, val_df = split_source(source_df, 0.2, cfg.seed)
    labels = sorted(source_df["label_raw"].unique())
    label_map = {v: i for i, v in enumerate(labels)}

    train_loader = make_loader(
        PalmDataset(train_df, cfg, label_map, training=True),
        cfg, shuffle=True, drop_last=False
    )
    val_loader = make_loader(
        PalmDataset(val_df, cfg, label_map, training=False),
        cfg, shuffle=False
    )

    model = SYEnet(len(label_map), cfg).to(DEVICE)
    if cfg.channels_last and DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)

    epochs = cfg.quick_epochs if cfg.quick_run else cfg.epochs

    criterion = nn.CrossEntropyLoss(
        label_smoothing=cfg.label_smoothing
    )
    scaler = make_grad_scaler(
        cfg.use_amp and DEVICE.type == "cuda"
    )

    ckpt = (
        Path(cfg.project_root)
        / f"syenet_{source_name.lower()}_{cfg.baseline_profile}.pt"
    )
    best_acc = -1.0
    best_loss = float("inf")
    history = []
    random_acc = 1.0 / len(label_map)

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()

        train_stats = run_epoch(
            model, train_loader, criterion,
            optimizer=optimizer, scaler=scaler,
            gradient_clip=cfg.gradient_clip
        )
        val_stats = run_epoch(
            model, val_loader, criterion,
            optimizer=None, scaler=scaler
        )

        if scheduler is not None:
            scheduler.step(val_stats["loss"])

        current_lr = optimizer.param_groups[0]["lr"]
        elapsed = time.perf_counter() - start

        row = {
            "epoch": epoch,
            "train_loss": train_stats["loss"],
            "train_acc": train_stats["top1"],
            "train_top5": train_stats["top5"],
            "val_loss": val_stats["loss"],
            "val_acc": val_stats["top1"],
            "val_top5": val_stats["top5"],
            "learning_rate": current_lr,
            "seconds": elapsed,
            "train_predicted_classes": train_stats["predicted_classes"],
            "val_predicted_classes": val_stats["predicted_classes"]
        }
        history.append(row)

        improved = (
            val_stats["top1"] > best_acc
            or (
                abs(val_stats["top1"] - best_acc) < 1e-12
                and val_stats["loss"] < best_loss
            )
        )

        if improved:
            best_acc = val_stats["top1"]
            best_loss = val_stats["loss"]
            torch.save({
                "model_state": model.state_dict(),
                "label_map": label_map,
                "best_val": best_acc,
                "best_val_loss": best_loss,
                "source": source_name,
                "feature_aggregation": cfg.feature_aggregation,
                "feature_dim": model.feature_dim,
                "baseline_profile": cfg.baseline_profile,
                "config": asdict(cfg)
            }, ckpt)

    del model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(history), ckpt


In [ ]:
history_tongji, ckpt_tongji = train_source(
    tongji_work_df, "Tongji", CFG
)

history_casia, ckpt_casia = train_source(
    casia_work_df, "CASIA", CFG
)


## 8. Feature extraction

In [ ]:
def load_model(ckpt_path, cfg):
    info = torch.load(
        ckpt_path, map_location=DEVICE,
        weights_only=False
    )

    saved_aggregation = info.get("feature_aggregation")
    if (
        saved_aggregation is not None
        and saved_aggregation != cfg.feature_aggregation
    ):
        raise ValueError(
            f"Checkpoint aggregation={saved_aggregation}, "
            f"CFG={cfg.feature_aggregation}."
        )

    model = SYEnet(len(info["label_map"]), cfg).to(DEVICE)
    model.load_state_dict(info["model_state"])
    if cfg.channels_last and DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()
    return model, info


model_from_tongji, info_tongji = load_model(ckpt_tongji, CFG)
model_from_casia, info_casia = load_model(ckpt_casia, CFG)


In [ ]:
@torch.no_grad()
def extract_all(model, df, cfg, noise_mode=None):
    labels = sorted(df["label_raw"].unique())
    label_map = {v: i for i, v in enumerate(labels)}
    dataset = PalmDataset(
        df, cfg, label_map,
        noise_mode=noise_mode,
        training=False
    )
    loader = make_loader(dataset, cfg, shuffle=False)

    features, numeric_labels, paths = [], [], []
    amp_enabled = cfg.use_amp and DEVICE.type == "cuda"

    model.eval()
    for patches, labels_batch, paths_batch in tqdm(loader):
        patches = patches.to(DEVICE, non_blocking=True)
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            feat = model.extract_features(patches)

        features.append(feat.cpu().numpy())
        numeric_labels.append(labels_batch.numpy())
        paths.extend(paths_batch)

    features = np.concatenate(features).astype(np.float32)
    numeric_labels = np.concatenate(numeric_labels).astype(np.int64)
    meta = df.reset_index(drop=True).copy()
    meta["label_numeric"] = numeric_labels
    return features, numeric_labels, paths, meta, label_map


casia_features, casia_labels, _, casia_meta, _ = extract_all(
    model_from_tongji, casia_work_df, CFG
)
tongji_features, tongji_labels, _, tongji_meta, _ = extract_all(
    model_from_casia, tongji_work_df, CFG
)


## 9. Few-shot palmprint recognition

In [ ]:
@torch.no_grad()
def crc_predict(train_features, train_labels, test_features, reg=1e-3, batch_size=256):
    X = torch.tensor(train_features, dtype=torch.float32, device=DEVICE).T
    Y = torch.tensor(test_features, dtype=torch.float32, device=DEVICE).T
    train_labels = np.asarray(train_labels)
    classes = np.unique(train_labels)

    d = X.shape[0]
    gram = X @ X.T + reg * torch.eye(d, device=DEVICE)
    projection = torch.linalg.solve(gram, X).T

    class_indices = {
        int(c): torch.tensor(
            np.where(train_labels == c)[0],
            dtype=torch.long, device=DEVICE
        )
        for c in classes
    }

    predictions = []
    for start in range(0, Y.shape[1], batch_size):
        query = Y[:, start:start + batch_size]
        alpha = projection @ query
        residuals = []
        for c in classes:
            idx = class_indices[int(c)]
            recon = X[:, idx] @ alpha[idx, :]
            residuals.append(torch.linalg.vector_norm(query - recon, dim=0))
        best = torch.stack(residuals).argmin(0).cpu().numpy()
        predictions.extend(classes[best].tolist())

    return np.asarray(predictions)

In [ ]:
def few_shot_split(labels, n, seed):
    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        if len(idx) <= n:
            continue
        idx = rng.permutation(idx)
        train_idx.extend(idx[:n].tolist())
        test_idx.extend(idx[n:].tolist())
    return np.asarray(train_idx), np.asarray(test_idx)

def repeated_experiment(features, labels, source, target, n, cfg):
    rows = []
    repeats = 2 if cfg.quick_run else cfg.repeats
    for repeat in range(repeats):
        train_idx, test_idx = few_shot_split(labels, n, cfg.seed + repeat)
        start = time.perf_counter()
        pred = crc_predict(
            features[train_idx], labels[train_idx],
            features[test_idx], cfg.crc_lambda
        )
        accuracy = accuracy_score(labels[test_idx], pred) * 100
        rows.append({
            "source_pretrain": source,
            "target": target,
            "protocol": "random_few_shot",
            "train_per_palm": n,
            "repeat": repeat + 1,
            "accuracy": accuracy,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "seconds": time.perf_counter() - start
        })

    df = pd.DataFrame(rows)
    return df

In [ ]:
def cross_session_tongji(features, meta, n, cfg):
    labels = meta["label_numeric"].to_numpy()
    sessions = meta["session"].to_numpy()

    s1 = np.where(sessions == "session1")[0]
    s2 = np.where(sessions == "session2")[0]

    if len(s1) == 0 or len(s2) == 0:
        raise ValueError("Tongji session1/session2 metadata is required.")

    rows = []
    repeats = 2 if cfg.quick_run else cfg.repeats

    for repeat in range(repeats):
        rng = np.random.default_rng(cfg.seed + repeat)
        gallery = []

        for c in np.unique(labels):
            candidates = s1[labels[s1] == c]
            if len(candidates) >= n:
                gallery.extend(
                    rng.choice(candidates, n, replace=False).tolist()
                )

        gallery = np.asarray(gallery)
        valid_classes = np.unique(labels[gallery])
        probe = s2[np.isin(labels[s2], valid_classes)]

        pred = crc_predict(
            features[gallery],
            labels[gallery],
            features[probe],
            cfg.crc_lambda
        )

        rows.append({
            "source_pretrain": "CASIA",
            "target": "Tongji",
            "protocol": "session1_to_session2",
            "train_per_palm": n,
            "repeat": repeat + 1,
            "accuracy": accuracy_score(labels[probe], pred) * 100,
            "n_train": len(gallery),
            "n_test": len(probe)
        })

    return pd.DataFrame(rows)


## 10. Example evaluation

In [ ]:
# Cross-dataset feature extraction
casia_features, casia_labels, _, casia_meta, _ = extract_all(
    model_from_tongji, casia_work_df, CFG
)

tongji_features, tongji_labels, _, tongji_meta, _ = extract_all(
    model_from_casia, tongji_work_df, CFG
)

# Few-shot recognition examples
casia_2shot = repeated_experiment(
    casia_features, casia_labels,
    source="Tongji", target="CASIA",
    n=2, cfg=CFG
)

tongji_2shot = repeated_experiment(
    tongji_features, tongji_labels,
    source="CASIA", target="Tongji",
    n=2, cfg=CFG
)
